# 01 - Exploration: KKBox Churn Prediction

## Step 1: Imports and setup

Just loading the libraries I need and setting a couple of display options.

- `pandas` (`pd`) - the main library for working with tables. A table = a DataFrame.
- `numpy` (`np`) - math/array library, pandas is built on top of it. Don't need it much yet.
- `Path` - makes it easier to point to files and folders.
- the two `set_option` lines just make tables print nicer (show all columns, wider output)
- `DATA` is a shortcut to my data folder, so I don't have to type the full path every time. The `..` means "go up one folder" - my notebook is inside `notebooks/`, so this points to `data/raw` in the project root.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA = Path("../data/raw")


In [2]:
for f in sorted(DATA.glob("*.csv")):
    size_mb = f.stat().st_size / 1e6
    print(f"{f.name:35s} {size_mb:>30.1f} MB")

members_v3.csv                                               427.9 MB
sample_submission_v2.csv                                      42.7 MB
sample_submission_zero.csv                                    45.6 MB
train.csv                                                     46.7 MB
train_v2.csv                                                  45.6 MB
transactions.csv                                            1729.3 MB
transactions_v2.csv                                          115.4 MB
user_logs.csv                                              30514.1 MB
user_logs_v2.csv                                            1431.5 MB


In [3]:
sorted(DATA.glob("*.csv"))[0].stat().st_size/1e6

427.921437

## Step 3: Look at the first few rows

Let's open the smaller files and see what's inside. Skipping the huge ones (`user_logs`, `transactions.csv`) for now, will deal with those later.

- `files` - list of file names I want to check
- `for fname in files` - loop through each one
- the `print` with `=` is just a separator so the output isn't one big mess
- `nrows=5` - only read the first 5 rows, no need to load the whole file
- `df.columns` - shows the column names
- `print(df)` - shows the actual rows of data


In [4]:
files = ["train_v2.csv", "members_v3.csv", "transactions_v2.csv", "sample_submission_v2.csv"]

for fname in files:
    print(f"\n{'='*60}\n{fname}\n{'='*60}")
    df = pd.read_csv(DATA / fname, nrows=5)
    print(f"Columns: {list(df.columns)}")
    print(df)



train_v2.csv
Columns: ['msno', 'is_churn']
                                           msno  is_churn
0  ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=         1
1  f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=         1
2  zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=         1
3  8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=         1
4  K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=         1

members_v3.csv
Columns: ['msno', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time']
                                           msno  city  bd  gender  registered_via  registration_init_time
0  Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=     1   0     NaN              11                20110911
1  +tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=     1   0     NaN               7                20110914
2  cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=     1   0     NaN              11                20110915
3  9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=     1   0     NaN              11 

## Step 4: Load the full files

Now loading the *whole* train, members, and transactions files (not just a few rows this time).

- `pd.read_csv(...)` - loads the entire file into a DataFrame
- `.shape` - tells me (rows, columns)
- `members_v3.csv` is around 428MB so this cell takes a bit longer to run


In [5]:
train = pd.read_csv(DATA / "train_v2.csv")
members = pd.read_csv(DATA / "members_v3.csv")
transactions = pd.read_csv(DATA / "transactions_v2.csv")

print(f"train:        {train.shape}")
print(f"members:      {members.shape}")
print(f"transactions: {transactions.shape}")


train:        (970960, 2)
members:      (6769473, 6)
transactions: (1431009, 9)


## Step 5: Check data types and memory usage

Checking what type each column is (number, text, etc.) and how much memory each table is using.

- `df.dtypes` - shows the type of each column (int, float, object/text)
- `df.memory_usage(deep=True)` - memory used per column. Need `deep=True` for text columns, otherwise pandas gives a way too small number
- dividing by `1e6` converts bytes to MB


In [6]:
for name, df in [("train", train), ("members", members), ("transactions", transactions)]:
    print(f"\n--- {name} ---")
    print(df.dtypes)
    print(df.memory_usage(deep=True)/1e6)
    print(f"Memory Total: {df.memory_usage(deep=True).sum()/ 1e6:.1f} MB")



--- train ---
msno        object
is_churn     int64
dtype: object
Index        0.000132
msno        90.299280
is_churn     7.767680
dtype: float64
Memory Total: 98.1 MB

--- members ---
msno                      object
city                       int64
bd                         int64
gender                    object
registered_via             int64
registration_init_time     int64
dtype: object
Index                       0.000132
msno                      629.560989
city                       54.155784
bd                         54.155784
gender                    268.051690
registered_via             54.155784
registration_init_time     54.155784
dtype: float64
Memory Total: 1114.2 MB

--- transactions ---
msno                      object
payment_method_id          int64
payment_plan_days          int64
plan_list_price            int64
actual_amount_paid         int64
is_auto_renew              int64
transaction_date           int64
membership_expire_date     int64
is_cancel        

## Step 6: Check for missing values

Checking how many values are missing (`NaN`) in each column.

- `df.isna()` - True/False for every cell, True = missing
- `.mean()` - turns that into a percentage (True counts as 1, False as 0)
- `.round(2)` - just rounds for readability


In [7]:
for name, df in [("train", train), ("members", members), ("transactions", transactions)]:
    print(f"\n--- {name} missing % ---")
    print((df.isna().mean() * 100).round(2))



--- train missing % ---
msno        0.0
is_churn    0.0
dtype: float64

--- members missing % ---
msno                       0.00
city                       0.00
bd                         0.00
gender                    65.43
registered_via             0.00
registration_init_time     0.00
dtype: float64

--- transactions missing % ---
msno                      0.0
payment_method_id         0.0
payment_plan_days         0.0
plan_list_price           0.0
actual_amount_paid        0.0
is_auto_renew             0.0
transaction_date          0.0
membership_expire_date    0.0
is_cancel                 0.0
dtype: float64


## Step 7: Check the target column (`is_churn`)

This is the thing we're trying to predict, so let's see how balanced it is.

- `value_counts()` - counts how many 0s and how many 1s
- `.mean()` - since the column is just 0s and 1s, this gives the churn rate directly
- if churn is rare (looks like ~9%), that's "class imbalance" and it'll matter a lot later for choosing metrics/models


In [8]:
print(train['is_churn'].value_counts())
print(f"\nChurn rate: {train['is_churn'].mean():.3%}")


is_churn
0    883630
1     87330
Name: count, dtype: int64

Churn rate: 8.994%


## Step 8: Look at the `members` table

Checking the demographic columns - `city`, `gender`, `registered_via`, and `bd` (age).

- `nunique()` - how many different values a column has
- `value_counts(dropna=False)` - also counts the missing values, not just the real ones
- `.to_dict()` - just makes it print a bit nicer
- `bd` min/max - checking for the weird age values we already know are in there (negative numbers, huge numbers)


In [9]:
print("Members:")
print(f"  city unique:           {members['city'].nunique()}")
print(f"  gender values:         {members['gender'].value_counts(dropna=False).to_dict()}")
print(f"  registered_via unique: {members['registered_via'].nunique()}")
print(f"  bd (age) range:        {members['bd'].min()} to {members['bd'].max()}")


Members:
  city unique:           21
  gender values:         {nan: 4429505, 'male': 1195355, 'female': 1144613}
  registered_via unique: 18
  bd (age) range:        -7168 to 2016


## Step 9: Check user overlap between tables

Later we'll join train + members + transactions together using `msno`. First let's check if the same users actually show up in each file.

- `set(...)` - turns a column into a set of unique values (no duplicates)
- `&` - users that are in both sets
- `-` - users in the first set but not the second
- this tells us how many users will end up with missing data after we join everything


In [10]:
train_users = set(train['msno'])
members_users = set(members['msno'])
trans_users = set(transactions['msno'])

print(f"train users:                 {len(train_users):,}")
print(f"members users:               {len(members_users):,}")
print(f"transactions users:          {len(trans_users):,}")
print(f"train ∩ members:             {len(train_users & members_users):,}")
print(f"train ∩ transactions:        {len(train_users & trans_users):,}")
print(f"train users missing members: {len(train_users - members_users):,}")


train users:                 970,960
members users:               6,769,473
transactions users:          1,197,050
train ∩ members:             860,967
train ∩ transactions:        933,578
train users missing members: 109,993


## Step 10: Fix the date columns

The date columns (like `registration_init_time`) are stored as plain numbers, e.g. `20170131`. Let's turn them into real dates.

- `pd.to_datetime(...)` - converts the column to an actual date type
- `format='%Y%m%d'` - tells it the order is year-month-day
- `errors='coerce'` - if something doesn't look like a valid date, just put `NaT` (missing date) instead of crashing
- `.min()` / `.max()` - quick check that the date range looks reasonable


In [11]:
# members
members['registration_init_time'] = pd.to_datetime(
    members['registration_init_time'], format='%Y%m%d', errors='coerce'
)
print("Registration date range:", members['registration_init_time'].min(), "→", members['registration_init_time'].max())

# transactions
for col in ['transaction_date', 'membership_expire_date']:
    transactions[col] = pd.to_datetime(transactions[col], format='%Y%m%d', errors='coerce')
print("Transaction date range:", transactions['transaction_date'].min(), "→", transactions['transaction_date'].max())


Registration date range: 2004-03-26 00:00:00 → 2017-04-29 00:00:00
Transaction date range: 2015-01-01 00:00:00 → 2017-03-31 00:00:00


## Step 11: Peek at `user_logs_v2`

This file is huge (1.4GB), and the other one (`user_logs.csv`) is even bigger (28GB). So instead of loading it all, let's just look at a small sample.

- `nrows=10000` - only read 10,000 rows
- `.shape` / `.dtypes` - check what the columns look like
- `.head()` - shows the first 5 rows. No `print()` needed - Jupyter shows the last line of a cell automatically.

That's it for this notebook. Next, we need to figure out how to deal with `user_logs` without running out of memory - that's where `polars` comes in.


In [12]:
user_logs_sample = pd.read_csv(DATA / "user_logs_v2.csv", nrows=10000)
print(user_logs_sample.shape)
print(user_logs_sample.dtypes)
user_logs_sample.head()


(10000, 9)
msno           object
date            int64
num_25          int64
num_50          int64
num_75          int64
num_985         int64
num_100         int64
num_unq         int64
total_secs    float64
dtype: object


,msno,date,num_25,num_50,num_75,num_985,num_100,num_unq,total_secs
0,u9E91QDTvHLq6NXjEaWv8u4QIqhrHk72kE+w31Gnhdg=,20170331,8,4,0,1,21,18,6309.273
1,nTeWW/eOZA/UHKdD5L7DEqKKFTjaAj3ALLPoAWsU8n0=,20170330,2,2,1,0,9,11,2390.699
2,2UqkWXwZbIjs03dHLU9KHJNNEvEkZVzm69f3jCS+uLI=,20170331,52,3,5,3,84,110,23203.337
3,ycwLc+m2O0a85jSLALtr941AaZt9ai8Qwlg9n0Nql5U=,20170331,176,4,2,2,19,191,7100.454
4,EGcbTofOSOkMmQyN1NMLxHEXJ1yV3t/JdhGwQ9wXjnI=,20170331,2,1,0,1,112,93,28401.558
